# Feature Engineering

Goal: create leakage-safe predictors for 60-minute-ahead household power forecasting.

Feature groups:
- current electrical measurements
- historical power lags
- calendar features
- rolling statistics
- 60-minute-ahead regression target

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

In [2]:
data_path = Path("../data/processed/household_power_validated.parquet")

df = pd.read_parquet(data_path)

df = df.sort_values("datetime").reset_index(drop=True)

print(df.shape)
print(df["datetime"].min())
print(df["datetime"].max())

(2075259, 10)
2006-12-16 17:24:00
2010-11-26 21:02:00


In [3]:
lag_minutes = [
    1,
    5,
    15,
    30,
    60,
    120,
    1380
]

for lag in lag_minutes:
    df[f"power_lag_{lag}"] = (
        df["Global_active_power"].shift(lag)
    )

In [4]:
df["rolling_mean_15"] = (
    df["Global_active_power"]
    .rolling(window=15)
    .mean()
)

df["rolling_mean_60"] = (
    df["Global_active_power"]
    .rolling(window=60)
    .mean()
)

df["rolling_std_60"] = (
    df["Global_active_power"]
    .rolling(window=60)
    .std()
)

df["rolling_mean_1440"] = (
    df["Global_active_power"]
    .rolling(window=1440)
    .mean()
)

In [5]:
df["power_target_60"] = (
    df["Global_active_power"].shift(-60)
)

In [6]:
feature_cols = [
    # Current electrical state
    "Global_active_power",
    "Global_reactive_power",
    "Voltage",
    "Global_intensity",
    "Sub_metering_1",
    "Sub_metering_2",
    "Sub_metering_3",

    # Lag features
    "power_lag_1",
    "power_lag_5",
    "power_lag_15",
    "power_lag_30",
    "power_lag_60",
    "power_lag_120",
    "power_lag_1380",

    # Calendar
    "hour",
    "day_of_week",
    "month",
    "is_weekend",

    # Rolling statistics
    "rolling_mean_15",
    "rolling_mean_60",
    "rolling_std_60",
    "rolling_mean_1440"
]

target_col = "power_target_60"

In [8]:
df["hour"] = df["datetime"].dt.hour
df["day_of_week"] = df["datetime"].dt.dayofweek
df["month"] = df["datetime"].dt.month

df["is_weekend"] = (
    df["day_of_week"] >= 5
).astype(int)

In [9]:
missing_summary = (
    df[feature_cols + [target_col]]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

missing_summary

rolling_mean_1440        124210
rolling_std_60            30080
rolling_mean_60           30080
power_lag_1380            27359
rolling_mean_15           26969
power_lag_120             26099
power_lag_60              26039
power_target_60           26039
power_lag_30              26009
power_lag_15              25994
power_lag_5               25984
power_lag_1               25980
Global_reactive_power     25979
Sub_metering_3            25979
Sub_metering_2            25979
Sub_metering_1            25979
Global_intensity          25979
Voltage                   25979
Global_active_power       25979
hour                          0
day_of_week                   0
month                         0
is_weekend                    0
dtype: int64

In [10]:
model_df = (
    df[
        ["datetime"]
        + feature_cols
        + [target_col]
    ]
    .dropna()
    .reset_index(drop=True)
)

In [11]:
print("Original rows:", len(df))
print("Usable modeling rows:", len(model_df))
print("Rows removed:", len(df) - len(model_df))

model_df.head()

Original rows: 2075259
Usable modeling rows: 1950244
Rows removed: 125015


,datetime,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,power_lag_1,power_lag_5,...,power_lag_1380,hour,day_of_week,month,is_weekend,rolling_mean_15,rolling_mean_60,rolling_std_60,rolling_mean_1440,power_target_60
0,2006-12-17 17:23:00,3.238,0.172,233.58,13.8,0.0,0.0,16.0,3.130,3.796,...,2.926,17,6,12,1,3.592667,3.552533,0.693240,2.487728,3.612
1,2006-12-17 17:24:00,3.390,0.170,232.65,14.6,0.0,0.0,17.0,3.238,3.808,...,3.452,17,6,12,1,3.579600,3.527967,0.671762,2.487154,3.612
2,2006-12-17 17:25:00,3.360,0.164,231.20,14.4,0.0,0.0,16.0,3.390,3.580,...,4.870,17,6,12,1,3.566533,3.502833,0.648586,2.485765,3.576
3,2006-12-17 17:26:00,3.400,0.162,232.86,14.6,0.0,0.0,17.0,3.360,3.370,...,4.868,17,6,12,1,3.552933,3.478233,0.623112,2.484394,3.552
4,2006-12-17 17:27:00,3.396,0.164,232.89,14.6,0.0,0.0,17.0,3.400,3.130,...,4.866,17,6,12,1,3.530800,3.454667,0.598129,2.483011,3.550


In [12]:
print(model_df.isna().sum().sum())

0


In [13]:
output_path = Path(
    "../data/processed/household_power_features.parquet"
)

model_df.to_parquet(
    output_path,
    index=False
)

print("Saved:", output_path)
print("Shape:", model_df.shape)

Saved: ../data/processed/household_power_features.parquet
Shape: (1950244, 24)


In [15]:
print("Original row count:", len(df))
print("Usable row count:", len(model_df))
print("Rows removed:", len(df) - len(model_df))
print("Final shape:", model_df.shape)

Original row count: 2075259
Usable row count: 1950244
Rows removed: 125015
Final shape: (1950244, 24)
